In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
from xgboost import XGBClassifier
from pulseiq.ml_logic.registry import save_model, load_model
from pulseiq.ml_logic.train import train_model


In [ ]:
!wget 'https://data.mendeley.com/datasets/m8cgwxs9s6/2/files/4c109a9f-2462-4dce-b93c-5789168c5401'

In [ ]:
#!unzip raw_data/example.zip

In [ ]:
path = "/home/davig/code/pulseIQ/raw_data/DiaBD_A Diabetes Dataset for Enhanced Risk Analysis and Research in Bangladesh (1).csv"
df = pd.read_csv(path)
df.head()

In [ ]:
df1 = df.copy()

In [ ]:
'''
Droping the ID columns
'''

df1 = df.drop(columns=['systolic_bp', 'diastolic_bp', 'family_diabetes', 'family_hypertension', 'glucose', 'stroke'])

In [ ]:
df1.head(2)

In [ ]:
'''
Binaryzing Diabetes to 0 = Normal and 1 = Diabetes
'''

map_diabetes = {
    'No': 0,
    'Yes': 1
}

df1['diabetic'] = df1['diabetic'].map(map_diabetes)

In [ ]:
'''
Binaryzing Sex to 0 = Female and 1 = Male
'''

map_sex = {
    'Female': 0,
    'Male': 1
}

df1['gender'] = df1['gender'].map(map_sex)

In [ ]:
df1.head(4)

In [ ]:
df1.info()

In [ ]:
df1.head(2)

In [ ]:

'''
Scanling the dataset
'''
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()




X_scaled = scaler.set_output(transform='pandas').fit_transform(df1[['age', 'pulse_rate', 'height', 'weight', 'bmi']])
y_diabetic = df1['diabetic']
y_hypertensive = df1['hypertensive']
y_cv = df1['cardiovascular_disease']


In [ ]:
y_diabetic

In [ ]:
sns.heatmap(pd.concat([X_scaled, y_diabetic], axis=1).corr(), annot=True)

## Logistic Regression baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.multiclass import OneVsRestClassifier

model = LogisticRegression(random_state=42)

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_scaled, y_diabetic, train_size=0.7, random_state=42, stratify=y_diabetic)

model.fit(X_train_d, y_train_d)

cv = cross_validate(model, X_train_d, y_train_d, cv=5, scoring=['roc_auc'])

cv['test_roc_auc'].mean()

In [ ]:
model = LogisticRegression(random_state=42)

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_scaled, y_hypertensive, train_size=0.7, random_state=42, stratify=y_hypertensive)

model.fit(X_train_h, y_train_h)

cv = cross_validate(model, X_train_h, y_train_h, cv=5, scoring=['roc_auc'])

cv['test_roc_auc'].mean()

In [ ]:
model = LogisticRegression(random_state=42)

X_train_cv, X_test_cv, y_train_cv, y_test_cv = train_test_split(X_scaled, y_cv, train_size=0.7, random_state=42, stratify=y_cv)

model.fit(X_train_cv, y_train_cv)

cv = cross_validate(model, X_train_cv, y_train_cv, cv=5, scoring=['roc_auc'])

cv['test_roc_auc'].mean()

In [ ]:
from sklearn.metrics import roc_auc_score
def model_f(y):
    params={
        'n_estimators': 400,
        'max_depth': 6,
        'learning_rate': 0.9964688888846461,
        'min_child_weight': 1, 'subsample': 0.5004689439033313,
        'colsample_bytree': 0.7613304427154312,
        'gamma': 0.054181861010946, 'reg_alpha': 0.05914008164898518,
        'reg_lambda': 1.1933199710131277e-06
    }
    model_xgb = XGBClassifier(**params)

    model_xgb.fit(X_scaled, y)

    cv_xgb = cross_validate(model_xgb, X_scaled, y, cv=5, scoring=['roc_auc'])

    skf = StratifiedKFold(n_splits=5)
    skf.get_n_splits()

    scores = []

    for i, (train_index, test_index) in enumerate(skf.split(X_scaled, y)):
        X_train = X_scaled.iloc[train_index]
        y_train = y.iloc[train_index]
        X_test = X_scaled.iloc[test_index]
        y_test = y.iloc[test_index]

        model_xgb = XGBClassifier(eval_metric='logloss')

        model_xgb.fit(X_train, y_train)

        y_pred = model_xgb.predict_proba(X_test)[:, 1]


        scores.append(roc_auc_score(y_test, y_pred))
    return np.mean(scores)

In [ ]:
scores_d = model_f(y_diabetic)
scores_h = model_f(y_hypertensive)
scores_cv = model_f(y_cv)

In [ ]:
scores_d, scores_h, scores_cv

In [41]:
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X = X_scaled
y = y_diabetic

def objective(trial):
    # clasificador
    classifier_name = trial.suggest_categorical('classifier', ['XGBClassifier']) #, 'LogisticRegression', 'RandomForestClassifier'

    if classifier_name == 'XGBClassifier':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 1, log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'eval_metric': 'logloss',
            'random_state': 42,
            'n_jobs': -1
        }


    # Kfolds
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    X_arr = np.array(X_scaled)
    y_arr = np.array(y_diabetic)

    for train_idx, val_idx in skf.split(X_arr, y_arr):
        X_train, X_test = X_arr[train_idx], X_arr[val_idx]
        y_train, y_test = y_arr[train_idx], y_arr[val_idx]

        #if classifier_name == 'XGBClassifier':
        model = XGBClassifier(**params)
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        #elif classifier_name == 'LogisticRegression':
        #    model = LogisticRegression(**params)
        #    model.fit(X_train, y_train)
        #else:
        #    model = RandomForestClassifier(**params)
        #    model.fit(X_train, y_train)

        preds = model.predict_proba(X_test)
        score = roc_auc_score(y_test, preds[:,1])
        fold_scores.append(score)

    return np.mean(fold_scores)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=200)

print("Mejor ROC_AUC:", study.best_value)
print("Mejores parámetros:", study.best_params)


[I 2026-08-27 14:31:03,569] A new study created in memory with name: no-name-ab44f56e-bc25-4359-b9ec-849bac58ebd6
[I 2026-08-27 14:31:09,569] Trial 0 finished with value: 0.663526980541674 and parameters: {'classifier': 'XGBClassifier', 'n_estimators': 900, 'max_depth': 7, 'learning_rate': 0.030829282911316026, 'min_child_weight': 1, 'subsample': 0.6349746136342845, 'colsample_bytree': 0.8114127059981662, 'gamma': 2.267168233177796e-06, 'reg_alpha': 0.007414604201436304, 'reg_lambda': 0.5303540029257122}. Best is trial 0 with value: 0.663526980541674.
[I 2026-08-27 14:31:14,417] Trial 1 finished with value: 0.695621428573854 and parameters: {'classifier': 'XGBClassifier', 'n_estimators': 1000, 'max_depth': 6, 'learning_rate': 0.01993037455942986, 'min_child_weight': 8, 'subsample': 0.510185010880013, 'colsample_bytree': 0.8652917562163986, 'gamma': 0.0001941156635666476, 'reg_alpha': 2.3445613188969907e-08, 'reg_lambda': 5.3217510095281416e-08}. Best is trial 1 with value: 0.6956214285

Mejor ROC_AUC: 0.7317883744230269
Mejores parámetros: {'classifier': 'XGBClassifier', 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.06978508228546695, 'min_child_weight': 1, 'subsample': 0.8321276547435521, 'colsample_bytree': 0.7953657295471905, 'gamma': 1.2924138770056416e-07, 'reg_alpha': 0.35555313508365, 'reg_lambda': 3.189114424075836e-07}


In [44]:
params = {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.06978508228546695, 'min_child_weight': 1, 'subsample': 0.8321276547435521, 'colsample_bytree': 0.7953657295471905, 'gamma': 1.2924138770056416e-07, 'reg_alpha': 0.35555313508365, 'reg_lambda': 3.189114424075836e-07}
model_t = train_model(XGBClassifier, X_scaled, y_diabetic, params)
save_model(model_t, 'diabetic')


Fold 0: AUC = 0.6696
Fold 1: AUC = 0.7327
Fold 2: AUC = 0.7522
Fold 3: AUC = 0.7484
Fold 4: AUC = 0.7194
Mean AUC: 0.7245


(PosixPath('/home/davig/code/pulseIQ/frontend/XGBoost_diabetic.json'),
 PosixPath('/home/davig/code/pulseIQ/frontend/config_diabetic.json'))

In [45]:
model_d = load_model('diabetic')

prediction = model_d.predict(X_test_d)

prediction.mean()

np.float64(0.000630119722747322)

## Hypertensive



In [35]:
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X = X_scaled
y = y_hypertensive

def objective(trial):
    # clasificador
    classifier_name = trial.suggest_categorical('classifier', ['XGBClassifier']) #, 'LogisticRegression', 'RandomForestClassifier'

    if classifier_name == 'XGBClassifier':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 1, log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'eval_metric': 'logloss',
            'random_state': 42,
            'n_jobs': -1
        }


    # Kfolds
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    X_arr = np.array(X_scaled)
    y_arr = np.array(y_hypertensive)

    for train_idx, val_idx in skf.split(X_arr, y_arr):
        X_train, X_test = X_arr[train_idx], X_arr[val_idx]
        y_train, y_test = y_arr[train_idx], y_arr[val_idx]

        #if classifier_name == 'XGBClassifier':
        model = XGBClassifier(**params)
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        #elif classifier_name == 'LogisticRegression':
        #    model = LogisticRegression(**params)
        #    model.fit(X_train, y_train)
        #else:
        #    model = RandomForestClassifier(**params)
        #    model.fit(X_train, y_train)

        preds = model.predict_proba(X_test)
        score = roc_auc_score(y_test, preds[:,1])
        fold_scores.append(score)

    return np.mean(fold_scores)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=200




               )

print("Mejor ROC_AUC:", study.best_value)
print("Mejores parámetros:", study.best_params)


[I 2026-08-27 14:22:02,681] Trial 4 finished with value: 0.7467621986429102 and parameters: {'classifier': 'XGBClassifier', 'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.019848846290883054, 'min_child_weight': 4, 'subsample': 0.7558638393435414, 'colsample_bytree': 0.7633147607435048, 'gamma': 0.8493051438293531, 'reg_alpha': 1.6848685095485638e-07, 'reg_lambda': 7.557524979906524e-05}. Best is trial 4 with value: 0.7467621986429102.
[I 2026-08-27 14:22:07,664] Trial 5 finished with value: 0.6884340138691784 and parameters: {'classifier': 'XGBClassifier', 'n_estimators': 600, 'max_depth': 8, 'learning_rate': 0.17703522938233432, 'min_child_weight': 8, 'subsample': 0.5800572049151687, 'colsample_bytree': 0.8899279911633722, 'gamma': 0.3444889655129962, 'reg_alpha': 6.2617927056023675e-06, 'reg_lambda': 0.02628349278691971}. Best is trial 4 with value: 0.7467621986429102.
[I 2026-08-27 14:22:15,482] Trial 6 finished with value: 0.6791612122511473 and parameters: {'classifier':

Mejor ROC_AUC: 0.7652737668079541
Mejores parámetros: {'classifier': 'XGBClassifier', 'n_estimators': 600, 'max_depth': 3, 'learning_rate': 0.010100986524636394, 'min_child_weight': 1, 'subsample': 0.6988948174004668, 'colsample_bytree': 0.7036209641032721, 'gamma': 6.693716298679582e-07, 'reg_alpha': 3.2877918020751725e-07, 'reg_lambda': 5.290022255752686e-05}


In [39]:
params = {'n_estimators': 600, 'max_depth': 3, 'learning_rate': 0.010100986524636394, 'min_child_weight': 1, 'subsample': 0.6988948174004668, 'colsample_bytree': 0.7036209641032721, 'gamma': 6.693716298679582e-07, 'reg_alpha': 3.2877918020751725e-07, 'reg_lambda': 5.290022255752686e-05}
model_t = train_model(XGBClassifier, X_scaled, y_hypertensive, params)
save_model(model_t, 'hypertensive')


Fold 0: AUC = 0.7217
Fold 1: AUC = 0.7713
Fold 2: AUC = 0.7344
Fold 3: AUC = 0.7767
Fold 4: AUC = 0.7729
Mean AUC: 0.7554


(PosixPath('/home/davig/code/pulseIQ/frontend/XGBoost_hypertensive.json'),
 PosixPath('/home/davig/code/pulseIQ/frontend/config_hypertensive.json'))

In [40]:
model_h = load_model('hypertensive')

prediction = model_h.predict(X_test_h)

prediction.mean()

np.float64(0.010081915563957152)

## Cardiovascular Disease

In [38]:
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X = X_scaled
y = y_cv

def objective(trial):
    # clasificador
    classifier_name = trial.suggest_categorical('classifier', ['XGBClassifier']) #, 'LogisticRegression', 'RandomForestClassifier'

    if classifier_name == 'XGBClassifier':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 1, log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'eval_metric': 'logloss',
            'random_state': 42,
            'n_jobs': -1
        }


    # Kfolds
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    X_arr = np.array(X_scaled)
    y_arr = np.array(y_cv)

    for train_idx, val_idx in skf.split(X_arr, y_arr):
        X_train, X_test = X_arr[train_idx], X_arr[val_idx]
        y_train, y_test = y_arr[train_idx], y_arr[val_idx]

        #if classifier_name == 'XGBClassifier':
        model = XGBClassifier(**params)
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        #elif classifier_name == 'LogisticRegression':
        #    model = LogisticRegression(**params)
        #    model.fit(X_train, y_train)
        #else:
        #    model = RandomForestClassifier(**params)
        #    model.fit(X_train, y_train)

        preds = model.predict_proba(X_test)
        score = roc_auc_score(y_test, preds[:,1])
        fold_scores.append(score)

    return np.mean(fold_scores)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=200)

print("Mejor ROC_AUC:", study.best_value)
print("Mejores parámetros:", study.best_params)


[I 2026-08-27 14:28:55,984] A new study created in memory with name: no-name-ed9f3c70-5d80-4b2d-bcb7-197bfb7c29c2
[I 2026-08-27 14:28:58,826] Trial 0 finished with value: 0.6876763991937083 and parameters: {'classifier': 'XGBClassifier', 'n_estimators': 800, 'max_depth': 6, 'learning_rate': 0.032941193947213364, 'min_child_weight': 10, 'subsample': 0.9323552699828954, 'colsample_bytree': 0.7444951291233266, 'gamma': 0.0025774649253744596, 'reg_alpha': 4.281866031467472, 'reg_lambda': 0.674809298916131}. Best is trial 0 with value: 0.6876763991937083.
[I 2026-08-27 14:29:04,662] Trial 1 finished with value: 0.6038753769962887 and parameters: {'classifier': 'XGBClassifier', 'n_estimators': 600, 'max_depth': 8, 'learning_rate': 0.12284525613106817, 'min_child_weight': 8, 'subsample': 0.9628202066871955, 'colsample_bytree': 0.7070682214340989, 'gamma': 0.021488570856026044, 'reg_alpha': 1.235091205360023e-08, 'reg_lambda': 0.00016258396677336432}. Best is trial 0 with value: 0.687676399193

KeyboardInterrupt: 

In [46]:
params =  {'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.010030187483214029, 'min_child_weight': 5, 'subsample': 0.812765113697849, 'colsample_bytree': 0.9417411410261993, 'gamma': 0.0007193844317030003, 'reg_alpha': 0.017635135956500642, 'reg_lambda': 0.008032239868452896}
model_t = train_model(XGBClassifier, X_scaled, y_cv, params)
save_model(model_t, 'cv')


Fold 0: AUC = 0.5608
Fold 1: AUC = 0.8034
Fold 2: AUC = 0.7202
Fold 3: AUC = 0.7341
Fold 4: AUC = 0.5858
Mean AUC: 0.6809


(PosixPath('/home/davig/code/pulseIQ/frontend/XGBoost_cv.json'),
 PosixPath('/home/davig/code/pulseIQ/frontend/config_cv.json'))

In [47]:
model_cv = load_model('cv')

prediction = model_cv.predict(X_test_cv)

prediction.mean()

np.float64(0.0)